In [1]:
import torch

In [ ]:
# https://www.sketchengine.eu/opus-parallel-corpora/: i ofund this intrestng


In [1]:
import argparse
import ast
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer
from transformers.models.marian import MarianMTModel

In [2]:
class TextDataset(Dataset):
    def __init__(self, tokenizer, original_data_path=None, text_data_list=None):
        self.tokenizer = tokenizer
        if original_data_path:
            self.df = pd.read_csv(original_data_path)
            self.text_data_list = self.df['text'].tolist()
            self.text_num_list = [1] * len(self.text_data_list)
        else:
            self.text_data_list = text_data_list
            self.text_num_list = [1] * len(text_data_list)
    
    def __len__(self):
        return len(self.text_data_list)
    
    def __getitem__(self, idx):
        return self.text_data_list[idx]
    
    def collate_fn(self, batch):
        return self.tokenizer(batch, return_tensors='pt', padding=True, truncation=True)

In [3]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, MarianMTModel
from tqdm import tqdm
import numpy as np

class BackTranslation:
    def __init__(self, lang="de", cache_dir=None):
        self.lang = lang
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.cache_dir = cache_dir  # For model caching
        
        try:
            print(f"Using device: {self.device}")
            # Load models with caching and move to device in one go
            model_kwargs = {"cache_dir": cache_dir} if cache_dir else {}
            self.tokenizers = {
                "en_lang": AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}", **model_kwargs),
                "lang_en": AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en", **model_kwargs)
            }
            self.models = {
                "en_lang": MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}", **model_kwargs).to(self.device),
                "lang_en": MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en", **model_kwargs).to(self.device)
            }
            # Enable FP16 if available
            if self.device.type == "cuda":
                self.models["en_lang"].half()
                self.models["lang_en"].half()
        except Exception as e:
            print(f"Warning: Could not load models for language {lang}: {str(e)}")
            self.tokenizers = {"en_lang": None}
            self.models = {}

    def _translate_batch(self, model, tokenizer, batch, temperature, **generate_kwargs):
        """Helper method to process translation batches efficiently"""
        with torch.no_grad():  # Disable gradient computation
            outputs = model.generate(
                **batch.to(self.device),
                temperature=temperature,
                **generate_kwargs
            )
        # Decode in batch instead of individually
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        return [text.strip() for text in decoded]

    def do_back_translation(self, original_data_path, batch_size, temperature, **generate_kwargs):
        if not self.tokenizers["en_lang"]:
            return None, None

        temp1, temp2 = (temperature[0], temperature[0]) if len(temperature) == 1 else temperature
        
        # Optimize DataLoader with pin_memory for CUDA
        dataset_kwargs = {"original_data_path": original_data_path}
        text_dataset = TextDataset(self.tokenizers["en_lang"], **dataset_kwargs)
        dataloader = DataLoader(
            text_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=4,
            pin_memory=(self.device.type == "cuda"),
            collate_fn=text_dataset.collate_fn
        )

        # Pre-allocate lists for better memory management
        text_num_list = text_dataset.text_num_list
        lang_out_list = []
        
        # First translation (en -> lang)
        for batch in tqdm(dataloader, desc=f"Translating to {self.lang}", leave=False):
            lang_out_list.extend(self._translate_batch(
                self.models["en_lang"],
                self.tokenizers["en_lang"],
                batch,
                temp1,
                **generate_kwargs
            ))

        # Second translation (lang -> en)
        lang_dataset = TextDataset(self.tokenizers["lang_en"], text_data_list=lang_out_list)
        dataloader = DataLoader(
            lang_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=4,
            pin_memory=(self.device.type == "cuda"),
            collate_fn=lang_dataset.collate_fn
        )

        en_out_list = []
        for batch in tqdm(dataloader, desc=f"Translating back from {self.lang}", leave=False):
            en_out_list.extend(self._translate_batch(
                self.models["lang_en"],
                self.tokenizers["lang_en"],
                batch,
                temp2,
                **generate_kwargs
            ))

        # Use numpy for faster array splitting
        en_out_array = np.array(en_out_list)
        text_augment_list = []
        start = 0
        for text_num in text_num_list:
            text_augment_list.append(en_out_array[start:start + text_num].tolist())
            start += text_num

        return lang_out_list, text_augment_list

    def __del__(self):
        """Clean up GPU memory"""
        if hasattr(self, "models"):
            for model in self.models.values():
                if model is not None:
                    del model
        torch.cuda.empty_cache() if self.device.type == "cuda" else None

In [5]:
def run_multi_language_pipeline(cache_dir="./model_cache"):
    languages = ["af", "sq", "ar", "hy", "eu", "bg", "bn", "ca", "zh", "hr", "cs", "da", "nl", "et", "fi", "fr", 
                 "gl", "ka", "de", "el", "gu", "ht", "he", "hi", "hu", "is", "id", "ga", "it", "ja", "kn", "kk", "km", 
                 "ko", "lv", "lt", "mk", "ms", "ml", "mt", "mr", "ne", "no", "fa", "pl", "pt", "pa", "ro", "ru", "sr", 
                 "sk", "sl", "es", "sw", "sv", "ta", "te", "th", "tr", "uk", "ur", "vi", "cy", "yi", "zu"]
    
    # Paths and parameters
    original_data_path = "dataset_Small/dataset_train.csv"
    output_path = "dataset_Small/dataset_aug_train_all_1.csv"
    batch_size = 150
    temperature = [1.0]
    num_beams = 5
    generate_kwargs = {"num_beams": num_beams, "do_sample": True}

    # Check for existing progress
    if os.path.exists(output_path):
        original_df = pd.read_csv(output_path)
        processed_langs = {col.split('_')[1] for col in original_df.columns if col.startswith('intermediate_')}
        languages = [lang for lang in languages if lang not in processed_langs]
        print(f"Resuming from existing file. Remaining languages: {len(languages)}")
    else:
        original_df = pd.read_csv(original_data_path)
        processed_langs = set()
    print(f"Dataset size: {len(original_df)}")

    # Process remaining languages with progress bar
    for lang in tqdm(languages, desc="Processing languages"):
        print(f"\nProcessing language: {lang}")
        
        try:
            bt = BackTranslation(lang=lang, cache_dir=cache_dir)
            intermediate_texts, augmented_texts = bt.do_back_translation(
                original_data_path=original_data_path,
                batch_size=batch_size,
                temperature=temperature,
                **generate_kwargs
            )
            
            if intermediate_texts and augmented_texts:
                original_df[f'intermediate_{lang}'] = intermediate_texts
                original_df[f'augment_{lang}'] = augmented_texts
            else:
                print(f"Skipping {lang} due to model loading failure")
                original_df[f'intermediate_{lang}'] = None
                original_df[f'augment_{lang}'] = None
            
            # Save with compression to save space
            original_df.to_csv(output_path, index=False, compression='gzip')
            print(f"Progress saved to {output_path} for language {lang}")
            
            # Clean up memory
            del bt
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            
        except Exception as e:
            print(f"Error processing {lang}: {str(e)}")
            original_df[f'intermediate_{lang}'] = None
            original_df[f'augment_{lang}'] = None
            original_df.to_csv(output_path, index=False, compression='gzip')

    print(f"\nFinal results saved to {output_path}")
    print("\nFinal Results (first few rows):")
    print(original_df.head())

In [ ]:
if __name__ == "__main__":
    # Set random seed
    torch.manual_seed(42)
    run_multi_language_pipeline()

Dataset size: 2


Processing languages:   0%|                              | 0/65 [00:00<?, ?it/s]


Processing language: af
Using device: cuda


/home/mtech/torch/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")

nslating to af: 100%|██████████████████████████| 1/1 [00:00<00:00,  2.20it/s]
                                                                             
Processing languages:   2%|▎                     | 1/65 [00:08<09:03,  8.48s/it]

Progress saved to dataset_Small/dataset_aug_train_all_1.csv for language af

Processing language: sq
Using device: cuda


...



ating to ru:   0%|                                  | 0/1 [00:00<?, ?it/s]

ating to ru: 100%|██████████████████████████| 1/1 [00:00<00:00,  4.70it/s]

                                                                          

ating back from ru:   0%|                           | 0/1 [00:00<?, ?it/s]

ating back from ru: 100%|███████████████████| 1/1 [00:00<00:00,  5.18it/s]

/tmp/ipykernel_144724/1230116241.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  original_df[f'augment_{lang}'] = None


Progress saved to dataset_Small/dataset_aug_train_all_1.csv for language ru

Processing language: sr
Using device: cuda
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
Skipping sr due to model loading failure
Progress saved to dataset_Small/dataset_aug_train_all_1.csv for language sr

Processing language: sk
Using device: cuda


/home/mtech/torch/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


ating to sk:   0%|                                  | 0/1 [00:00<?, ?it/s]

ating to sk: 100%|██████████████████████████| 1/1 [00:00<00:00,  4.95it/s]

                                                                          

ating back from sk:   0%|                           | 0/1 [00:00<?, ?it/s]

ating back from sk: 100%|███████████████████| 1/1 [00:00<00:00,  5.08it/s]

/tmp/ipykernel_144724/1230116241.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  original_df[f'intermediate_{lang}'] = intermediate_texts
/tmp/ipykernel_144724/1230116241.p

Progress saved to dataset_Small/dataset_aug_train_all_1.csv for language sk

Processing language: sl
Using device: cuda
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
Skipping sl due to model loading failure
Progress saved to dataset_Small/dataset_aug_train_all_1.csv for language sl

Processing language: es
Using device: cuda


/home/mtech/torch/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
